# 구글렌즈 + 키워드 검색 — 가격 책정 레퍼런스 실측

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

LAB-21(`가격 책정 레퍼런스 비교`)에서 FE와 논의하며 나온 아이디어 중 "구글렌즈 + 키워드
검색" 항목을 실측한다. 이미 설계 문서 `.claude/plans/ai/CH005-google-serp-grade-reference.md`
가 이 아이디어를 두 방식으로 쪼개 뒀다.

- **방식 A** — `google_lens` 매물(title/price/source/link)을 가격 구간별로 모아 히스토그램
  ("이 가격대에 매물이 N건")으로 쓴다. 기존 `CompsSource`에 새 백엔드로 얹는 모양.
- **방식 B** — `google` 검색의 "AI 개요"가 이미 계산해 주는 **등급별(A+/A/B/C) 가격대**를
  그대로 파싱한다. `pricer.py`의 `condition_coefficient()`가 지금 흉내 내는 임의 계수를
  실측 데이터로 대체할 잠재력이 방식 A보다 크다(CH005 "권장(잠정)").

이 노트북은 두 방식을 각각 실험 1(방식 B)·실험 2(방식 A)로 실측해서 커버리지를 잰다 —
CH002(`comps_sources.py`)가 시세 수집 백엔드 커버리지를 잰 것과 같은 성격이다. **가격
정확도가 아니라 "이 경로로 등급별/구간별 시세를 실제로 얻을 수 있는가"만 본다.**

## ⚠️ 실행 전에 반드시 읽을 것 — 유료 서드파티 API

- SerpApi는 **GMS/네이버와 달리 완전히 유료인 서드파티 스크레이핑 대행 서비스**다(Free
  250건/월, 이후 Starter $25/월부터). `AI/CLAUDE.md` 하드 룰 — "호출 비용·쿼터가 있는
  외부 API는 먼저 물어본다" — 대상이고, CH005 문서 자체가 "Claude는 이 실행을 하지
  않는다"고 명시해 뒀다.
- **이 노트북은 Claude가 작성만 했고 실행(셀 실행)하지 않았다.** 쿼터·비용이 드는 셀은
  전부 사용자가 직접 실행해야 한다.
- 실행하려면 `AI/.env`에 `SERPAPI_KEY`를 채운다(`.env.example`에 자리만 추가해 뒀다).
  없으면 아래 셀들이 조용히 건너뛰지 않고 **명시적으로 실패**한다 — 키 없이 무료 호출로
  착각하는 것을 막기 위해서다.
- 쿼리 10개(실험 1) + 이미지 3장(실험 2) 정도면 Free 티어 250건 안에서 충분히 끝난다.
  쿼리 수를 늘리기 전에 먼저 소량으로 커버리지부터 확인할 것.

## 0. 준비 — 키 로딩

`validation/PRICE/pricing-model-check/gms_client.py`의 `get_key()`를 그대로 재사용한다 —
`AI/.env`를 읽는 최소 파서가 이미 있는데 이 노트북에서 새로 만들 이유가 없다. `SERPAPI_KEY`는
그 파일이 모르는 키지만 `get_key(name)`은 임의의 환경변수 이름을 받으므로 그대로 쓸 수 있다.

In [ ]:
import sys
from pathlib import Path

_PRICING_CHECK_DIR = (Path.cwd().parent / "pricing-model-check").resolve()
if str(_PRICING_CHECK_DIR) not in sys.path:
    sys.path.insert(0, str(_PRICING_CHECK_DIR))

from gms_client import get_key  # noqa: E402 — sys.path 조정 이후 import

SERPAPI_KEY = get_key("SERPAPI_KEY")
if not SERPAPI_KEY:
    raise RuntimeError(
        "SERPAPI_KEY가 없습니다. AI/.env 에 채운 뒤 커널을 재시작하세요. "
        "(무료 티어라도 명시적으로 키를 넣는 게 의도적 행위여야 합니다 — AI/CLAUDE.md)"
    )
print(f"SERPAPI_KEY 로드됨 (길이 {len(SERPAPI_KEY)}자)")

In [ ]:
from dataclasses import dataclass, field

SERPAPI_BASE = "https://serpapi.com/search.json"


@dataclass
class SerpApiResult:
    """SerpApi 호출 1건의 원 결과 + 걸린 시간. 판정은 각 실험 셀에서 한다."""

    query: str
    latency_ms: float = 0.0
    status_code: int = 0
    payload: dict = field(default_factory=dict)
    error: str = ""

    @property
    def ok(self) -> bool:
        return not self.error and self.status_code == 200


def serpapi_get(params: dict, *, timeout: float = 30.0) -> SerpApiResult:
    """SerpApi에 GET 1회. 네트워크·HTTP 계층만 — 파싱은 호출부가 한다."""
    import time

    import httpx

    query = params.get("q", "")
    started = time.perf_counter()
    try:
        resp = httpx.get(SERPAPI_BASE, params={**params, "api_key": SERPAPI_KEY}, timeout=timeout)
    except Exception as exc:  # noqa: BLE001 — 스파이크 스크립트, 원인 그대로 노출
        return SerpApiResult(query, error=f"{type(exc).__name__}: {exc}")
    elapsed = (time.perf_counter() - started) * 1000

    if resp.status_code != 200:
        return SerpApiResult(
            query, elapsed, resp.status_code, error=f"HTTP {resp.status_code} {resp.text[:200]}"
        )
    return SerpApiResult(query, elapsed, resp.status_code, payload=resp.json())

## 1. 실험 1 (방식 B) — `engine=google`의 AI 개요 등급별 시세

CH005가 인용한 예시(`"Levis LVC 54501 시세"` 검색 시 A+/A/B/C 등급별 가격대)를 재현할 수
있는지 확인한다. 쿼리셋은 CH002(`comps_sources.py`)의 `DEFAULT_QUERIES`와 같은 구성 원칙을
따른다 — **유명 브랜드 5건 + 니치 5건**을 섞어야 "AI 개요가 유명 브랜드에서만 나온다" 같은
차등이 있는지 볼 수 있다.

`ai_overview` 필드가 즉시 오면(`text_blocks`) 그대로 파싱하고, `state: "deferred"`로 오면
`page_token`으로 후속 요청이 필요하다(CH005 §조사 결과) — 이번 실험은 그 두 번째 요청까지
포함한다.

In [ ]:
# CH002 DEFAULT_QUERIES와 같은 원칙(유명 브랜드 5 + 니치 5). 실제 검증 시 CH005 예시대로
# "시세" 키워드를 붙여 등급별 요약을 유도한다 — 상품명만 검색하면 일반 검색 결과만 나올 수 있다.
GRADE_REF_QUERIES = [
    "리바이스 501 빅E 데님 시세",
    "나이키 빈티지 스우시 후드 집업 시세",
    "폴로 랄프로렌 옥스포드 셔츠 빈티지 시세",
    "칼하트 디트로이트 자켓 빈티지 시세",
    "스톤아일랜드 니트 가디건 시세",
    "90년대 무지 워시드 데님 자켓 시세",
    "빈티지 체크 플란넬 셔츠 오버핏 시세",
    "구제 코듀로이 와이드 팬츠 시세",
    "빈티지 카레지 니트 베스트 시세",
    "구제 워크웨어 헤링본 코트 시세",
]


def fetch_ai_overview(query: str) -> dict:
    """`engine=google` 1회(+ deferred면 `engine=google_ai_overview` 후속 1회).

    반환 모양(판정용): {"query", "state", "text_blocks", "latency_ms", "error"}.
    `state`는 "missing"(ai_overview 필드 자체가 없음) / "complete" / "deferred_resolved" /
    "deferred_unresolved" / "error" 중 하나.
    """
    first = serpapi_get({"engine": "google", "q": query, "hl": "ko", "gl": "kr"})
    if not first.ok:
        return {"query": query, "state": "error", "text_blocks": [], "latency_ms": 0.0,
                "error": first.error}

    overview = first.payload.get("ai_overview")
    if not overview:
        return {"query": query, "state": "missing", "text_blocks": [],
                "latency_ms": first.latency_ms, "error": ""}

    if overview.get("text_blocks"):
        return {"query": query, "state": "complete", "text_blocks": overview["text_blocks"],
                "latency_ms": first.latency_ms, "error": ""}

    page_token = overview.get("page_token")
    if not page_token:
        return {"query": query, "state": "missing", "text_blocks": [],
                "latency_ms": first.latency_ms, "error": ""}

    # deferred — 실제 필드명(page_token 파라미터 이름 등)은 최초 실행 시 응답을 찍어 보고
    # SerpApi 문서 기준과 다르면 여기를 고친다.
    second = serpapi_get({"engine": "google_ai_overview", "page_token": page_token})
    total_latency = first.latency_ms + second.latency_ms
    if not second.ok:
        return {"query": query, "state": "deferred_unresolved", "text_blocks": [],
                "latency_ms": total_latency, "error": second.error}
    blocks = second.payload.get("ai_overview", {}).get("text_blocks", [])
    return {"query": query, "state": "deferred_resolved" if blocks else "deferred_unresolved",
            "text_blocks": blocks, "latency_ms": total_latency, "error": ""}

In [ ]:
import re

# CH005 스크린샷 예시의 등급 라벨 패턴 — 우리 ConditionGrade(S/A/B/C)와 이름이 다르다는 점에
# 유의(CH005 "아키텍처가 걸리는 지점" 참고). 여기서는 매칭 여부만 확인하고 매핑은 하지 않는다.
_GRADE_LABEL_RE = re.compile(r"(A\+|A|B|C)\s*급")
_PRICE_RANGE_RE = re.compile(r"([\d,]+)\s*원\s*~\s*([\d,]+)\s*원")


def extract_grade_bands(text_blocks: list[dict]) -> list[dict]:
    """text_blocks에서 "X급: N원 ~ M원" 패턴을 뽑는다. 못 찾으면 빈 리스트."""
    bands = []
    for block in text_blocks:
        candidates = [block.get("snippet", "")]
        for item in block.get("list", []) or []:
            candidates.append(item.get("snippet", ""))
        for text in candidates:
            grade_match = _GRADE_LABEL_RE.search(text)
            price_match = _PRICE_RANGE_RE.search(text)
            if grade_match and price_match:
                bands.append({
                    "grade": grade_match.group(1),
                    "min": int(price_match.group(1).replace(",", "")),
                    "max": int(price_match.group(2).replace(",", "")),
                    "raw": text,
                })
    return bands


ai_overview_results = [fetch_ai_overview(q) for q in GRADE_REF_QUERIES]

print("=== 실험 1 — engine=google AI 개요 커버리지 ===")
state_counts: dict[str, int] = {}
for result in ai_overview_results:
    state_counts[result["state"]] = state_counts.get(result["state"], 0) + 1
    bands = extract_grade_bands(result["text_blocks"]) if result["text_blocks"] else []
    tag = f"등급 {len(bands)}개 파싱" if bands else result["state"]
    print(f"  {result['query']:<28} → {tag} ({result['latency_ms']:.0f}ms)")
    for band in bands:
        print(f"      {band['grade']}급: {band['min']:,}원 ~ {band['max']:,}원")
    if result["error"]:
        print(f"      오류: {result['error']}")

print("\n=== state 분포 ===")
for state, count in state_counts.items():
    print(f"  {state}: {count}/{len(ai_overview_results)}")
print(
    "\n판정 기준: 'missing'이 대부분이면 방식 B는 커버리지가 낮은 것 — CH005 '범위 밖' 항목대로"
    "\n방식 A(히스토그램)만 검토하는 쪽으로 기운다. 'complete'/'deferred_resolved'가 유의미하게"
    "\n나오면 등급 라벨(A+/A/B/C)이 우리 ConditionGrade(S/A/B/C)와 어떻게 대응할지가 다음 질문이다."
)

## 2. 실험 2 (방식 A) — `engine=google_lens` 이미지 + 키워드 매물

사용자가 말한 "구글렌즈 + 키워드 검색"과 정확히 같은 조합이다(CH005). 이미지 URL이 필수라
`TEST_IMAGES`에 **공개적으로 접근 가능한 이미지 URL**을 넣어야 한다 — 지금 채워 둔 것은
자리표시자(placeholder)다. 실제 검증은 실제 상품 사진(가려도 되는 예시 매물 사진)의
공개 URL로 바꿔서 돌려야 의미가 있다.

응답의 `visual_matches`(또는 이미지 종류에 따라 `products`)에서 title/price/source/link를
뽑아 기존 `CompItem`과 같은 모양으로 정규화한다 — 채택되면 `registry.COMPS_BACKENDS`에
`"google_lens"` 키로 얹는 그림과 바로 이어진다(CH005 §방식 A).

In [ ]:
# TODO(사용자): placeholder URL을 실제 공개 상품 이미지 URL로 교체할 것.
# (image_url, keyword) 쌍 — keyword는 build_query()가 만드는 것과 같은 성격의 검색어.
TEST_IMAGES = [
    ("https://example.com/placeholder-1.jpg", "리바이스 501 빅E 데님"),
    ("https://example.com/placeholder-2.jpg", "나이키 빈티지 스우시 후드 집업"),
    ("https://example.com/placeholder-3.jpg", "구제 워크웨어 헤링본 코트"),
]


@dataclass
class LensCompItem:
    """google_lens 매물 1건 — app/pricing/schemas.py의 CompItem과 같은 모양으로 맞춘다."""

    title: str
    price: int
    source: str
    url: str = ""


def fetch_google_lens(image_url: str, keyword: str) -> dict:
    """`engine=google_lens` 1회. 반환: {"keyword", "items", "raw_section", "latency_ms", "error"}.

    `raw_section`은 실제로 어느 필드(visual_matches/products)에서 파싱했는지 기록한다 —
    이미지 종류에 따라 둘 중 뭐가 채워지는지가 아직 실측 전이라 확인용으로 남긴다.
    """
    result = serpapi_get({"engine": "google_lens", "url": image_url, "q": keyword,
                            "hl": "ko", "gl": "kr"})
    if not result.ok:
        return {"keyword": keyword, "items": [], "raw_section": "", "latency_ms": 0.0,
                "error": result.error}

    for section_name in ("visual_matches", "products"):
        rows = result.payload.get(section_name)
        if not rows:
            continue
        items = []
        for row in rows:
            price = row.get("price", {})
            extracted = price.get("extracted_value") if isinstance(price, dict) else None
            if extracted is None:
                continue
            items.append(LensCompItem(
                title=str(row.get("title", ""))[:200],
                price=int(extracted),
                source=row.get("source", "google_lens"),
                url=str(row.get("link", "")),
            ))
        if items:
            return {"keyword": keyword, "items": items, "raw_section": section_name,
                    "latency_ms": result.latency_ms, "error": ""}

    return {"keyword": keyword, "items": [], "raw_section": "", "latency_ms": result.latency_ms,
            "error": ""}

In [ ]:
import statistics

lens_results = [fetch_google_lens(url, keyword) for url, keyword in TEST_IMAGES]

print("=== 실험 2 — engine=google_lens 매물 커버리지 ===")
for result in lens_results:
    count = len(result["items"])
    status = f"{count}건 ({result['raw_section'] or '없음'})" if not result["error"] else f"실패: {result['error']}"
    print(f"  {result['keyword']:<28} → {status} ({result['latency_ms']:.0f}ms)")
    for item in result["items"][:5]:
        print(f"      {item.price:>10,}원  {item.title[:40]}  [{item.source}]")

all_prices = [item.price for r in lens_results for item in r["items"]]
if all_prices:
    print(f"\n표본 {len(all_prices)}건 — 중앙값 {statistics.median(all_prices):,.0f}원, "
          f"최소 {min(all_prices):,}원, 최대 {max(all_prices):,}원")

    # LAB-21이 언급한 "x축 가격범위, y축 등록물품수" 그래프의 원시 데이터 — 구간 폭은
    # 표본이 늘면 조정한다. 지금은 3구간짜리 간이 히스토그램으로 형태만 확인한다.
    bucket_width = max(1, (max(all_prices) - min(all_prices)) // 3) or 1
    buckets: dict[int, int] = {}
    for price in all_prices:
        bucket = (price - min(all_prices)) // bucket_width
        buckets[bucket] = buckets.get(bucket, 0) + 1
    print("\n가격 구간별 매물 수(간이 히스토그램):")
    for bucket, count in sorted(buckets.items()):
        low = min(all_prices) + bucket * bucket_width
        high = low + bucket_width
        print(f"  {low:>10,}원 ~ {high:>10,}원 : {'■' * count} ({count}건)")
else:
    print("\n매물을 하나도 못 얻었다 — placeholder 이미지 URL을 실제 URL로 바꿨는지 먼저 확인할 것.")

---

## 결론 (실행 후 채울 것)

| # | 항목 | 근거 셀 | 결과 |
| --- | --- | --- | --- |
| 1 | AI 개요(`ai_overview`) 커버리지 — 몇 %의 쿼리에서 나오는가 | §1 state 분포 | |
| 2 | 등급 라벨(A+/A/B/C)이 실제로 파싱되는가 | §1 | |
| 3 | 우리 `ConditionGrade`(S/A/B/C)와 등급 라벨을 어떻게 매핑할지 | §1 | |
| 4 | `google_lens` 매물 표본 수·품질(중고 비율 확인 여부) | §2 | |
| 5 | `visual_matches` vs `products` 중 실제로 채워지는 필드 | §2 `raw_section` | |
| 6 | 지연시간 — LLM 웹검색(CH002: 7.7초)과 비교해 동기 가정에 영향 있는가 | §1, §2 | |
| 7 | 방식 A/B 중 채택(또는 병행) 판단 | 전체 | |

### 다음 단계

- 결과를 바탕으로 CH005 문서의 "다음 단계 제안" 2·3번(방식 결정 → 인터페이스/스키마 설계)을
  진행한다. **신규 도메인 로직이므로 그 설계 문서도 구현 전에 사용자 승인이 먼저다**
  (`AI/CLAUDE.md`).
- 이 노트북에서 쓴 실제 요청 수(무료 티어 250건 중 몇 건 소모했는지)를 CH005나 노션
  LAB-21/LAB-47과 같은 성격의 검증 기록으로 남겨 둘 것.
- `TEST_IMAGES`를 placeholder 그대로 실행하면 전부 매물 0건으로 나온다 — 반드시 실제
  공개 이미지 URL로 교체 후 실행할 것.